# Liver snRNA-seq Age Bin Analysis

## 분석 목표
1. age >= 30인 세포 필터링
2. 6개 age bin으로 분류: -30, 31-40, 41-50, 51-60, 71-80, 81-
3. Age bin별 세포 수 barplot
4. Age bin별 celltype stacked proportion barplot
5. Age bin에 따른 gene expression Spearman correlation 분석

---
## 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scanpy as sc
from scipy.stats import spearmanr
from scipy.stats import pearsonr

sc.settings.verbosity = 1

---
## 2. Load adata

In [ ]:
# adata 경로를 실제 파일 경로로 수정하세요
adata = sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad")
print(f"전체 세포 수: {adata.n_obs:,}, 유전자 수: {adata.n_vars:,}")
adata

In [ ]:
# obs 컬럼 확인
adata.obs.head()

In [ ]:
# age 컬럼 분포 확인
print("age 컬럼 유형:", adata.obs['age'].dtype)
print("age 기술통계:")
print(adata.obs['age'].describe())
print("\nage unique 값:")
print(sorted(adata.obs['age'].dropna().unique()))

---
## 3. Age filtering (>= 30) & Age Binning

In [ ]:
# age 컬럼 파싱 함수
# - 'Nan' 문자열 → np.nan
# - '31-40' 같은 범위 문자열 → 중간값 (e.g. 35.5)
# - '32' 같은 숫자 문자열 → float
def parse_age(val):
    val = str(val).strip()
    if val.lower() == 'nan' or val == '':
        return np.nan
    if '-' in val:
        parts = val.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return np.nan
    try:
        return float(val)
    except ValueError:
        return np.nan

adata.obs['age_numeric'] = adata.obs['age'].apply(parse_age)

print("파싱 결과 샘플 (원본 age → age_numeric):")
print(
    adata.obs[['age', 'age_numeric']]
    .drop_duplicates()
    .sort_values('age_numeric')
    .to_string()
)

# 'Nan' 및 NaN 제거 후 age >= 30 필터
adata = adata[adata.obs['age_numeric'].notna()].copy()
#adata = adata[adata.obs['age_numeric'] >= 30].copy()
print(f"\nage >= 30 필터 후: {adata.n_obs:,} 세포")

In [ ]:
# age 컬럼 분포 확인
print("age 컬럼 유형:", adata.obs['age_numeric'].dtype)
print("age 기술통계:")
print(adata.obs['age'].describe())
print("\nage unique 값:")
print(sorted(adata.obs['age_numeric'].dropna().unique()))

In [ ]:
# 7개의 age bin 정의: -30, 31-40, 41-50, 51-60, 61-70, 71-80, 81-
# age_numeric 기준으로 binning (범위값은 중간값 사용)
age = adata.obs['age_numeric'].astype(float)

conditions = [
    age <= 30,
    (age > 30) & (age <= 40),
    (age > 40) & (age <= 50),
    (age > 50) & (age <= 60),
    (age > 60) & (age <= 70),
    (age > 70) & (age <= 80),
    age > 80
]
bin_labels = ['-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-']

adata.obs['age_bin'] = np.select(conditions, bin_labels, default='unknown')

# unknown 확인 (파싱 오류 등)
n_unknown = (adata.obs['age_bin'] == 'unknown').sum()
if n_unknown > 0:
    print(f"[경고] 어느 bin에도 해당하지 않는 세포: {n_unknown:,}개 → 제외")
    adata = adata[adata.obs['age_bin'] != 'unknown'].copy()

# age_bin을 Categorical로 지정하여 순서 유지
adata.obs['age_bin'] = pd.Categorical(
    adata.obs['age_bin'],
    categories=bin_labels,
    ordered=True
)

print(f"최종 세포 수: {adata.n_obs:,}")
print("\nage_bin별 세포 수:")
print(adata.obs['age_bin'].value_counts().sort_index())

In [ ]:
# age_bin numeric 값 (correlation 분석용)
bin_numeric_map = {label: i+1 for i, label in enumerate(bin_labels)}
adata.obs['age_bin_numeric'] = adata.obs['age_bin'].map(bin_numeric_map).astype(float)
print("age_bin -> numeric 매핑:", bin_numeric_map)

---
## 4. Age bin별 세포 수 Barplot

In [ ]:
adata.obs

In [ ]:
cell_counts = adata.obs['age_bin'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    cell_counts.index.astype(str),
    cell_counts.values,
    color=sns.color_palette('Blues_d', n_colors=len(cell_counts)),
    edgecolor='black',
    linewidth=0.7
)

# 각 bar 위에 세포 수 표시
for bar, count in zip(bars, cell_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(cell_counts.values) * 0.01,
        f'{count:,}',
        ha='center', va='bottom', fontsize=10
    )

ax.set_xlabel('Age Bin', fontsize=12)
ax.set_ylabel('Number of Cells', fontsize=12)
ax.set_title('Cell Count by Age Bin', fontsize=14)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Age bin별 Sample 수 barplot
sample_counts = (
    adata.obs.groupby('age_bin', observed=True)['sample']
    .nunique()
    .reindex(bin_labels)
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    sample_counts.index.astype(str),
    sample_counts.values,
    color=sns.color_palette('Greens_d', n_colors=len(sample_counts)),
    edgecolor='black',
    linewidth=0.7
)

for bar, count in zip(bars, sample_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(sample_counts.values) * 0.01,
        f'{count:,}',
        ha='center', va='bottom', fontsize=10
    )

ax.set_xlabel('Age Bin', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Sample Count by Age Bin', fontsize=14)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Age bin별 Disease_status에 따른 세포 수 stacked barplot
disease_cell_counts = (
    adata.obs.groupby(['age_bin', 'Disease_status'], observed=True)
    .size()
    .reset_index(name='count')
)
disease_cell_wide = disease_cell_counts.pivot_table(
    index='age_bin',
    columns='Disease_status',
    values='count',
    fill_value=0
).reindex(bin_labels)

disease_palette = sns.color_palette('tab10', n_colors=disease_cell_wide.shape[1])

fig, ax = plt.subplots(figsize=(10, 5))
disease_cell_wide.plot(
    kind='bar',
    stacked=True,
    color=disease_palette,
    ax=ax,
    edgecolor='white',
    linewidth=0.3,
    width=0.7
)

ax.set_xlabel('Age Bin', fontsize=12)
ax.set_ylabel('Number of Cells', fontsize=12)
ax.set_title('Cell Count by Age Bin and Disease Status', fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Disease Status', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9, title_fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Age bin별 Disease_status에 따른 Sample 수 stacked barplot
disease_sample_counts = (
    adata.obs.groupby(['age_bin', 'Disease_status'], observed=True)['sample']
    .nunique()
    .reset_index(name='n_samples')
)
disease_sample_wide = disease_sample_counts.pivot_table(
    index='age_bin',
    columns='Disease_status',
    values='n_samples',
    fill_value=0
).reindex(bin_labels)

fig, ax = plt.subplots(figsize=(10, 5))
disease_sample_wide.plot(
    kind='bar',
    stacked=True,
    color=disease_palette,
    ax=ax,
    edgecolor='white',
    linewidth=0.3,
    width=0.7
)

ax.set_xlabel('Age Bin', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Sample Count by Age Bin and Disease Status', fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Disease Status', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9, title_fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Disease_group 컬럼 생성: MASL + MASH → MASLD, Healthy → Healthy, 나머지 제외
disease_map = {
    'Healthy': 'Healthy',
    'MASL':    'MASLD',
    'MASH':    'MASLD',
}
adata.obs['Disease_group'] = adata.obs['Disease_status'].map(disease_map)

# Healthy / MASLD 이외 제외
n_before = adata.n_obs
adata = adata[adata.obs['Disease_group'].isin(['Healthy', 'MASLD'])].copy()
print(f"Healthy/MASLD 이외 제외: {n_before - adata.n_obs:,}개 세포")
print("\nDisease_group 분포:")
print(adata.obs['Disease_group'].value_counts())

---
## 5. Age bin별 Celltype Stacked Proportion Barplot

In [ ]:
# celltype 컬럼명 확인 (실제 컬럼명에 맞게 수정하세요)
celltype_col = 'celltype'

# Healthy / MASLD 서브셋
adata_healthy = adata[adata.obs['Disease_group'] == 'Healthy'].copy()
adata_masld   = adata[adata.obs['Disease_group'] == 'MASLD'].copy()
print(f"Healthy: {adata_healthy.n_obs:,} 세포")
print(f"MASLD:   {adata_masld.n_obs:,} 세포")
print(f"\ncelltype 수: {adata.obs[celltype_col].nunique()}")
print(adata.obs[celltype_col].value_counts())

In [ ]:
# 비율/세포수 테이블 계산 함수
def calc_prop_count(adata_sub, celltype_col, bin_labels):
    grp = (
        adata_sub.obs.groupby(['age_bin', celltype_col], observed=True)
        .size()
        .reset_index(name='count')
    )
    grp['proportion'] = grp.groupby('age_bin', observed=True)['count'].transform(lambda x: x / x.sum())
    prop_wide = grp.pivot_table(index='age_bin', columns=celltype_col,
                                values='proportion', fill_value=0).reindex(bin_labels)
    count_wide = grp.pivot_table(index='age_bin', columns=celltype_col,
                                 values='count', fill_value=0).reindex(bin_labels)
    return prop_wide, count_wide

prop_healthy, count_healthy = calc_prop_count(adata_healthy, celltype_col, bin_labels)
prop_masld,   count_masld   = calc_prop_count(adata_masld,   celltype_col, bin_labels)

# 모든 celltype 통합 palette (두 그룹 컬럼 union)
all_celltypes = sorted(set(prop_healthy.columns) | set(prop_masld.columns))
palette_map = {ct: c for ct, c in zip(all_celltypes, sns.color_palette('tab20', n_colors=len(all_celltypes)))}
prop_healthy = prop_healthy.reindex(columns=all_celltypes, fill_value=0)
prop_masld   = prop_masld.reindex(columns=all_celltypes, fill_value=0)
count_healthy = count_healthy.reindex(columns=all_celltypes, fill_value=0)
count_masld   = count_masld.reindex(columns=all_celltypes, fill_value=0)
palette = [palette_map[ct] for ct in all_celltypes]
print("계산 완료")

In [ ]:
# Celltype Stacked Proportion Barplot — Healthy vs MASLD
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

for ax, prop_wide, title in zip(axes, [prop_healthy, prop_masld], ['Healthy', 'MASLD']):
    prop_wide.plot(
        kind='bar', stacked=True, color=palette,
        ax=ax, edgecolor='white', linewidth=0.3, width=0.7, legend=False
    )
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Age Bin', fontsize=12)
    ax.set_ylabel('Proportion', fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    sns.despine(ax=ax)

# 공통 legend
handles = [plt.Rectangle((0, 0), 1, 1, color=palette_map[ct]) for ct in all_celltypes]
fig.legend(handles, all_celltypes, title='Celltype',
           bbox_to_anchor=(1.01, 0.95), loc='upper left', fontsize=9, title_fontsize=10)
fig.suptitle('Celltype Proportion by Age Bin', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Celltype Count Stacked Barplot — Healthy vs MASLD
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, count_wide, title in zip(axes, [count_healthy, count_masld], ['Healthy', 'MASLD']):
    count_wide.plot(
        kind='bar', stacked=True, color=palette,
        ax=ax, edgecolor='white', linewidth=0.3, width=0.7, legend=False
    )
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Age Bin', fontsize=12)
    ax.set_ylabel('Number of Cells', fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    sns.despine(ax=ax)

handles = [plt.Rectangle((0, 0), 1, 1, color=palette_map[ct]) for ct in all_celltypes]
fig.legend(handles, all_celltypes, title='Celltype',
           bbox_to_anchor=(1.01, 0.95), loc='upper left', fontsize=9, title_fontsize=10)
fig.suptitle('Cell Count by Age Bin', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 5-1. Age Bin별 Celltype Proportion 유의성 검정

Stacked proportion barplot에서 눈으로 확인한 변화를 통계적으로 검정합니다.

- **방법**: Sample 단위 proportion → Kruskal-Wallis test (비모수) → BH FDR 보정  
- **유의 기준**: padj < 0.05  
- 유의한 celltype에 대해 boxplot + stripplot으로 시각화합니다.

In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

def test_celltype_proportion_change(adata_sub, celltype_col, bin_order, group_name):
    """
    Sample 단위 celltype proportion을 계산하고
    age bin 간 Kruskal-Wallis test로 유의성 검정.
    반환: (결과 DataFrame, sample_prop DataFrame)
    """
    obs = adata_sub.obs.copy()
    obs = obs[obs['age_bin'].isin(bin_order)].copy()

    # sample × celltype count
    sample_ct = (
        obs.groupby(['sample', 'age_bin', celltype_col], observed=True)
        .size().reset_index(name='count')
    )
    sample_total = (
        obs.groupby(['sample', 'age_bin'], observed=True)
        .size().reset_index(name='total')
    )
    sample_prop = sample_ct.merge(sample_total, on=['sample', 'age_bin'])
    sample_prop['proportion'] = sample_prop['count'] / sample_prop['total']
    sample_prop['age_bin'] = pd.Categorical(
        sample_prop['age_bin'], categories=bin_order, ordered=True
    )

    celltypes_all = sorted(obs[celltype_col].dropna().unique())
    results = []
    for ct in celltypes_all:
        ct_data = sample_prop[sample_prop[celltype_col] == ct]
        bin_groups = [
            ct_data[ct_data['age_bin'] == b]['proportion'].values
            for b in bin_order
        ]
        bin_groups = [g for g in bin_groups if len(g) >= 2]
        if len(bin_groups) < 2:
            continue
        try:
            stat, pval = kruskal(*bin_groups)
        except Exception:
            stat, pval = np.nan, np.nan
        results.append({'celltype': ct, 'H_statistic': stat, 'pval': pval,
                         'n_samples': ct_data['sample'].nunique()})

    df_res = pd.DataFrame(results)
    if len(df_res) > 0:
        _, padj, _, _ = multipletests(df_res['pval'].fillna(1), method='fdr_bh')
        df_res['pval_adj'] = padj
        df_res['significant'] = df_res['pval_adj'] < 0.05
        df_res = df_res.sort_values('pval_adj').reset_index(drop=True)

    print(f"\n{'='*50}")
    print(f"[{group_name}] Celltype Proportion Kruskal-Wallis Test")
    print(df_res.to_string(index=False))
    return df_res, sample_prop


# Healthy / MASLD 각각 검정 (Section 5에서 생성된 adata_healthy, adata_masld 사용)
prop_test_results = {}
sample_props = {}

for group_name, adata_g in [('Healthy', adata_healthy), ('MASLD', adata_masld)]:
    df_test, sp_df = test_celltype_proportion_change(
        adata_g, celltype_col, bin_labels, group_name
    )
    prop_test_results[group_name] = df_test
    sample_props[group_name] = sp_df

In [ ]:
def plot_significant_proportion(
    sample_prop, df_test, celltype_col, bin_order, group_name, padj_cutoff=1
):
    """
    유의한 celltype(padj < cutoff)에 대해
    age bin별 proportion boxplot + stripplot 시각화.
    """
    sig_cts = df_test[df_test['pval_adj'] < padj_cutoff]['celltype'].tolist()
    if not sig_cts:
        print(f"[{group_name}] 유의한 celltype 없음 (padj < {padj_cutoff})")
        return

    n_ct = len(sig_cts)
    ncols = min(4, n_ct)
    nrows = (n_ct + ncols - 1) // ncols
    palette_bins = sns.color_palette('coolwarm', len(bin_order))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    for i, ct in enumerate(sig_cts):
        ax = axes[i]
        ct_data = sample_prop[sample_prop[celltype_col] == ct].copy()
        row = df_test[df_test['celltype'] == ct].iloc[0]

        sns.boxplot(
            data=ct_data, x='age_bin', y='proportion',
            palette=palette_bins, order=bin_order,
            showfliers=False, ax=ax, linewidth=0.8
        )
        sns.stripplot(
            data=ct_data, x='age_bin', y='proportion',
            order=bin_order, color='black', size=3,
            ax=ax, alpha=0.6, jitter=True
        )
        ax.set_title(
            f"{ct}\nH={row['H_statistic']:.2f}, padj={row['pval_adj']:.3f}",
            fontsize=9, fontweight='bold'
        )
        ax.set_xlabel('Age Bin', fontsize=8)
        ax.set_ylabel('Proportion', fontsize=8)
        ax.tick_params(axis='x', rotation=45, labelsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    fig.suptitle(
        f'[{group_name}] Significant Celltype Proportion Changes (padj<{padj_cutoff})',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()


for group in ['Healthy', 'MASLD']:
    plot_significant_proportion(
        sample_props[group], prop_test_results[group],
        celltype_col, bin_labels, group
    )

In [ ]:
# Heatmap: -log10(padj) by celltype — Healthy vs MASLD 비교
def plot_proportion_significance_heatmap(prop_test_results, padj_cutoff=0.1):
    import matplotlib.colors as mcolors

    # 두 그룹 merge
    dfs = []
    for group, df in prop_test_results.items():
        df = df[['celltype', 'pval_adj']].copy()
        df.columns = ['celltype', group]
        dfs.append(df.set_index('celltype'))
    hm = pd.concat(dfs, axis=1)

    # -log10(padj) 변환 (0 → 0, NaN → 0)
    hm_log = -np.log10(hm.clip(lower=1e-300).fillna(1))
    sig_line = -np.log10(padj_cutoff)  # 유의성 기준선

    # celltype을 Healthy -log10(padj) 기준 정렬
    sort_col = 'Healthy' if 'Healthy' in hm_log.columns else hm_log.columns[0]
    hm_log = hm_log.sort_values(sort_col, ascending=False)

    fig, ax = plt.subplots(figsize=(5, max(4, len(hm_log) * 0.45)))
    sns.heatmap(
        hm_log, cmap='YlOrRd', ax=ax,
        linewidths=0.4, linecolor='lightgray',
        cbar_kws={'label': '-log10(padj)'},
        annot=True, fmt='.2f', annot_kws={'size': 8}
    )
    # 유의 기준선 표시
    ax.axvline(x=0, color='white', linewidth=0)
    ax.set_title(
        f'Celltype Proportion Change Significance\n'
        f'(-log10 padj, dashed = {padj_cutoff} cutoff)',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Disease Group', fontsize=11)
    ax.set_ylabel('Celltype', fontsize=11)
    plt.tight_layout()
    plt.show()

    # 요약 출력
    print(f"\n유의 기준: padj < {padj_cutoff}")
    for group in prop_test_results:
        sig = prop_test_results[group][prop_test_results[group]['pval_adj'] < padj_cutoff]['celltype'].tolist()
        print(f"[{group}] 유의한 celltype ({len(sig)}개): {sig}")


plot_proportion_significance_heatmap(prop_test_results)

---
## 6. Age Correlation Gene Analysis

### 6-1. 전체 세포 - 실제 age 값 기반 Spearman correlation

In [ ]:
# Correlation 분석용 age bin 필터: '-30' 과 '81-' 제외
exclude_bins = ['-30', '81-']
corr_bin_labels = [b for b in bin_labels if b not in exclude_bins]
print(f"Correlation 분석 age bin: {corr_bin_labels}")

# Disease_group별 필터된 서브셋 준비
groups_corr = {
    'Healthy': adata_healthy[~adata_healthy.obs['age_bin'].isin(exclude_bins)].copy(),
    'MASLD':   adata_masld[~adata_masld.obs['age_bin'].isin(exclude_bins)].copy(),
}

expr_dict = {}
ages_dict = {}

for group, adata_g in groups_corr.items():
    expr = adata_g.X
    if not isinstance(expr, np.ndarray):
        expr = expr.toarray()
    expr_dict[group] = expr
    ages_dict[group] = adata_g.obs['age_numeric'].values
    print(f"{group} — {adata_g.n_obs:,} 세포, age: {ages_dict[group].min():.0f}~{ages_dict[group].max():.0f}")

In [ ]:
# Healthy / MASLD 각각 age vs gene expression Spearman correlation
# (age bin: 31-40, 41-50, 51-60, 61-70, 71-80 만 포함)
age_corr_results = {}

for group, adata_g in groups_corr.items():
    print(f"[{group}] correlation 계산 중...")
    expr = expr_dict[group]
    ages = ages_dict[group]
    var_names = adata_g.var_names

    corrs, pvals = [], []
    for i in range(expr.shape[1]):
        r, p = spearmanr(ages, expr[:, i])
        corrs.append(r)
        pvals.append(p)

    df_corr = pd.DataFrame({'gene': var_names, 'spearman_r': corrs, 'pval': pvals})
    age_corr_results[group] = df_corr

    df_pos = df_corr[(df_corr['spearman_r'] > 0) & (df_corr['pval'] < 0.05)]
    df_neg = df_corr[(df_corr['spearman_r'] < 0) & (df_corr['pval'] < 0.05)]
    print(f"  significant: pos={len(df_pos)}, neg={len(df_neg)}")
    print(f"  Top 5 positive:\n{df_pos.nlargest(5, 'spearman_r')[['gene','spearman_r','pval']].to_string(index=False)}")
    print(f"  Top 5 negative:\n{df_neg.nsmallest(5, 'spearman_r')[['gene','spearman_r','pval']].to_string(index=False)}\n")

In [ ]:
# 결과 저장
output_dir = "/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/02_analysis/"
for group, df_corr in age_corr_results.items():
    path = output_dir + f"Nucseq_age_correlation_{group}.csv"
    df_corr.to_csv(path, index=False)
    print(f"저장 완료: {path}")

### 6-2. Age bin numeric 기반 Spearman correlation

In [ ]:
# age_bin_numeric 기반 Spearman correlation — Healthy / MASLD
# (age bin: 31-40, 41-50, 51-60, 61-70, 71-80 만 포함)
bin_corr_results = {}

for group, adata_g in groups_corr.items():
    print(f"[{group}] age_bin correlation 계산 중...")
    expr = expr_dict[group]
    age_bin_num = adata_g.obs['age_bin_numeric'].values
    var_names = adata_g.var_names

    corrs_bin, pvals_bin = [], []
    for i in range(expr.shape[1]):
        r, p = spearmanr(age_bin_num, expr[:, i])
        corrs_bin.append(r)
        pvals_bin.append(p)

    df_bin = pd.DataFrame({'gene': var_names, 'spearman_r': corrs_bin, 'pval': pvals_bin})
    bin_corr_results[group] = df_bin

    df_pos = df_bin[(df_bin['spearman_r'] > 0) & (df_bin['pval'] < 0.05)]
    df_neg = df_bin[(df_bin['spearman_r'] < 0) & (df_bin['pval'] < 0.05)]
    print(f"  significant: pos={len(df_pos)}, neg={len(df_neg)}")
    print(f"  Top 5 positive:\n{df_pos.nlargest(5, 'spearman_r')[['gene','spearman_r','pval']].to_string(index=False)}")
    print(f"  Top 5 negative:\n{df_neg.nsmallest(5, 'spearman_r')[['gene','spearman_r','pval']].to_string(index=False)}\n")

In [ ]:
for group, df_bin in bin_corr_results.items():
    path = output_dir + f"Nucseq_age_bin_correlation_{group}.csv"
    df_bin.to_csv(path, index=False)
    print(f"저장 완료: {path}")

### 6-3. Celltype별 Age Correlation

In [ ]:
celltypes = sorted(adata.obs[celltype_col].dropna().unique().tolist())
print(f"분석할 celltype ({len(celltypes)}개): {celltypes}")

In [ ]:
# celltype별 age correlation — Healthy / MASLD 각각
# (age bin: 31-40, 41-50, 51-60, 61-70, 71-80 만 포함)
celltype_corr_results = {'Healthy': {}, 'MASLD': {}}

for group, adata_g in groups_corr.items():
    print(f"\n{'='*40}\n[{group}] celltype별 correlation 계산")
    for ct in celltypes:
        adata_ct = adata_g[adata_g.obs[celltype_col] == ct].copy()
        n_cells = adata_ct.n_obs
        if n_cells < 30:
            print(f"  [SKIP] {ct}: {n_cells}개 (< 30)")
            continue
        print(f"  [분석] {ct}: {n_cells:,}개")

        expr_ct = adata_ct.X
        if not isinstance(expr_ct, np.ndarray):
            expr_ct = expr_ct.toarray()
        ages_ct = adata_ct.obs['age_numeric'].values

        corrs_ct, pvals_ct = [], []
        for i in range(expr_ct.shape[1]):
            r, p = spearmanr(ages_ct, expr_ct[:, i])
            corrs_ct.append(r)
            pvals_ct.append(p)

        celltype_corr_results[group][ct] = pd.DataFrame({
            'gene': adata_ct.var_names,
            'spearman_r': corrs_ct,
            'pval': pvals_ct
        })

print("\n분석 완료!")

In [ ]:
# Healthy / MASLD 각각 Top correlation gene 확인
for group in ['Healthy', 'MASLD']:
    print(f"\n{'='*50}\n[{group}]")
    for ct, df_ct in celltype_corr_results[group].items():
        df_sig = df_ct[df_ct['pval'] < 0.05]
        n_pos = (df_sig['spearman_r'] > 0).sum()
        n_neg = (df_sig['spearman_r'] < 0).sum()
        print(f"\n  {ct}: sig={len(df_sig)} (pos:{n_pos}, neg:{n_neg})")
        print("  Top 5 pos:", df_sig[df_sig['spearman_r'] > 0].nlargest(5, 'spearman_r')['gene'].tolist())
        print("  Top 5 neg:", df_sig[df_sig['spearman_r'] < 0].nsmallest(5, 'spearman_r')['gene'].tolist())

In [ ]:
# celltype별 결과 저장 — Healthy / MASLD 각각
for group in ['Healthy', 'MASLD']:
    for ct, df_ct in celltype_corr_results[group].items():
        safe_ct = ct.replace('/', '_').replace(' ', '_')
        path = output_dir + f"Nucseq_age_correlation_{group}_{safe_ct}.csv"
        df_ct.to_csv(path, index=False)
        print(f"저장: {path}")

In [ ]:
# ============================================================
# 6-3 통합 Excel 저장: Healthy / MASLD × 각 celltype 결과를
# 하나의 Excel 파일로 통합 (pval<0.05 & r>0 → 빨강, pval<0.05 & r<0 → 파랑)
# ============================================================
import glob
import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter
output_dir = "/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/02_analysis/"
RED_FILL  = PatternFill(start_color="FF9999", end_color="FF9999", fill_type="solid")
BLUE_FILL = PatternFill(start_color="99BBFF", end_color="99BBFF", fill_type="solid")

excel_path = output_dir + "Nucseq_age_correlation_celltype_integrated.xlsx"
wb = Workbook()
wb.remove(wb.active)  # 기본 빈 시트 제거

for group in ['Healthy', 'MASLD']:
    pattern = output_dir + f"Nucseq_age_correlation_{group}_*.csv"
    files = sorted(glob.glob(pattern))
    print(f"[{group}] {len(files)}개 파일 발견")

    all_dfs = []
    for f in files:
        basename = os.path.basename(f)
        # 파일명에서 celltype 추출: Nucseq_age_correlation_{group}_{ct}.csv
        ct = basename.replace(f"Nucseq_age_correlation_{group}_", "").replace(".csv", "")
        df = pd.read_csv(f)
        df.insert(0, 'celltype', ct)
        all_dfs.append(df)

    if not all_dfs:
        print(f"  [{group}] 파일 없음, 건너뜀")
        continue

    combined = pd.concat(all_dfs, ignore_index=True)

    ws = wb.create_sheet(title=group)

    # 헤더 작성
    headers = list(combined.columns)
    ws.append(headers)
    for cell in ws[1]:
        cell.font = Font(bold=True)

    # 데이터 행 작성 + 색상 적용
    for _, row in combined.iterrows():
        ws.append(list(row))
        current_row = ws.max_row
        pval  = row['pval']
        r_val = row['spearman_r']
        if pval < 0.05:
            fill = RED_FILL if r_val > 0 else (BLUE_FILL if r_val < 0 else None)
            if fill:
                for cell in ws[current_row]:
                    cell.fill = fill

    # 열 너비 자동 조정
    for col_idx, col_name in enumerate(headers, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = max(len(str(col_name)) + 4, 14)

    print(f"  [{group}] {len(combined)}행 작성 완료")

wb.save(excel_path)
print(f"\n저장 완료: {excel_path}")


### 6-4. Age bin별 평균 발현량 기반 correlation 시각화

In [ ]:
# 각 그룹의 top correlation 유전자를 시각화 (age_bin별 평균 발현)
for group, adata_g in groups_corr.items():
    df_corr = age_corr_results[group]
    top_pos = df_corr[(df_corr['spearman_r'] > 0) & (df_corr['pval'] < 0.05)].nlargest(5, 'spearman_r')['gene'].tolist()
    top_neg = df_corr[(df_corr['spearman_r'] < 0) & (df_corr['pval'] < 0.05)].nsmallest(5, 'spearman_r')['gene'].tolist()
    genes_of_interest = [g for g in top_pos + top_neg if g in adata_g.var_names]
    print(f"[{group}] 시각화 유전자: {genes_of_interest}")

In [ ]:
# age_bin별 평균 발현 line plot — Healthy / MASLD 각각
# (age bin: 31-40, 41-50, 51-60, 61-70, 71-80 만 포함)
for group, adata_g in groups_corr.items():
    df_corr = age_corr_results[group]
    top_pos = df_corr[(df_corr['spearman_r'] > 0) & (df_corr['pval'] < 0.05)].nlargest(5, 'spearman_r')['gene'].tolist()
    top_neg = df_corr[(df_corr['spearman_r'] < 0) & (df_corr['pval'] < 0.05)].nsmallest(5, 'spearman_r')['gene'].tolist()
    genes_of_interest = [g for g in top_pos + top_neg if g in adata_g.var_names]

    if not genes_of_interest:
        print(f"[{group}] 시각화할 significant 유전자 없음")
        continue

    adata_genes = adata_g[:, genes_of_interest].copy()
    expr_g = adata_genes.X
    if not isinstance(expr_g, np.ndarray):
        expr_g = expr_g.toarray()

    mean_df = pd.DataFrame(expr_g, index=adata_g.obs.index, columns=genes_of_interest)
    mean_df['age_bin'] = adata_g.obs['age_bin'].values
    mean_by_bin = mean_df.groupby('age_bin', observed=True)[genes_of_interest].mean().reindex(corr_bin_labels)

    n_genes = len(genes_of_interest)
    ncols = min(5, n_genes)
    nrows = (n_genes + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).flatten()

    for ax, gene in zip(axes, genes_of_interest):
        ax.plot(mean_by_bin.index.astype(str), mean_by_bin[gene],
                marker='o', color='steelblue' if group == 'Healthy' else 'tomato', linewidth=2)
        ax.set_title(gene, fontsize=11, fontweight='bold')
        ax.set_xlabel('Age Bin', fontsize=9)
        ax.set_ylabel('Mean Expr', fontsize=9)
        ax.tick_params(axis='x', rotation=30)
        sns.despine(ax=ax)

    for ax in axes[n_genes:]:
        ax.set_visible(False)

    fig.suptitle(f'[{group}] Mean Expression by Age Bin', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

### 6-5. Correlation heatmap: celltype x age_bin (선택 유전자 기준)

In [ ]:
# Spearman r heatmap: celltype x gene — Healthy / MASLD 각각
for group in ['Healthy', 'MASLD']:
    ct_results = celltype_corr_results[group]
    if len(ct_results) < 2:
        print(f"[{group}] celltype 수 부족, heatmap 생략")
        continue

    top_genes_all = set()
    for ct, df_ct in ct_results.items():
        top = df_ct[(df_ct['spearman_r'] > 0) & (df_ct['pval'] < 0.05)].nlargest(10, 'spearman_r')['gene'].tolist()
        top_genes_all.update(top)

    heatmap_data = {
        ct: df_ct.set_index('gene')['spearman_r'].reindex(list(top_genes_all))
        for ct, df_ct in ct_results.items()
    }
    heatmap_df = pd.DataFrame(heatmap_data).dropna(how='all')

    fig, ax = plt.subplots(figsize=(max(8, len(ct_results) * 1.5), max(6, len(heatmap_df) * 0.4)))
    sns.heatmap(
        heatmap_df, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
        ax=ax, linewidths=0.3, linecolor='lightgray',
        cbar_kws={'label': 'Spearman r'}
    )
    ax.set_title(f'[{group}] Age-Gene Correlation by Celltype', fontsize=13)
    ax.set_xlabel('Celltype', fontsize=11)
    ax.set_ylabel('Gene', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# groups_corr (Healthy / MASLD, corr_bin_labels 필터 적용) 저장
save_dir = "/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/01_Data/"

for group, adata_g in groups_corr.items():
    path = save_dir + f"Nucseq_{group}_corr_bins.h5ad"
    adata_g.write_h5ad(path)
    print(f"저장 완료: {path}  ({adata_g.n_obs:,} cells)")

---
## 7. 특정 유전자의 Age Bin별 Celltype 발현 Boxplot

관심 유전자(gene_of_interest)를 설정하면 Healthy / MASLD 각각에서  
celltype별로 age bin에 따른 발현량 boxplot과 Spearman r을 함께 표시합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 7을 독립 실행할 경우 아래 셀로 데이터를 불러오세요.
# Section 6까지 이미 실행된 상태라면 이 셀은 건너뜁니다.
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.stats import spearmanr

save_dir    = "/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/01_Data/"
celltype_col = 'celltype'

adata_healthy_corr = sc.read_h5ad(save_dir + "Nucseq_Healthy_corr_bins.h5ad")
adata_masld_corr   = sc.read_h5ad(save_dir + "Nucseq_MASLD_corr_bins.h5ad")

groups_corr = {
    'Healthy': adata_healthy_corr,
    'MASLD':   adata_masld_corr,
}

# corr_bin_labels 재정의 (저장 시 적용된 필터와 동일)
bin_labels      = ['-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-']
corr_bin_labels = [b for b in bin_labels if b not in ['-30', '81-']]

print("로드 완료")
for group, adata_g in groups_corr.items():
    print(f"  [{group}] {adata_g.n_obs:,} cells, bins: {sorted(adata_g.obs['age_bin'].unique())}")

In [ ]:
# ============================================================
# ★ 분석할 유전자를 여기에 입력하세요
gene_of_interest = 'SLC7A5'   # <-- 원하는 유전자명으로 변경
# ============================================================

In [ ]:
import scipy.sparse as sp

def plot_gene_celltype_boxplot(adata_g, gene, celltype_col, bin_order, title_prefix):
    """
    지정 유전자의 age bin별 발현 boxplot을 celltype별로 표시.
    각 subplot 제목에 Spearman r / p-value 표기.
    """
    if gene not in adata_g.var_names:
        print(f"[ERROR] '{gene}' 은 var_names에 없습니다.")
        return

    # 발현량 추출
    idx = list(adata_g.var_names).index(gene)
    expr_vec = adata_g.X[:, idx]
    if sp.issparse(expr_vec):
        expr_vec = np.array(expr_vec.todense()).flatten()
    else:
        expr_vec = np.array(expr_vec).flatten()

    # age bin 필터
    sub_obs = adata_g.obs.copy()
    sub_obs['_expr'] = expr_vec
    sub_obs = sub_obs[sub_obs['age_bin'].isin(bin_order)].copy()
    sub_obs['age_bin'] = pd.Categorical(sub_obs['age_bin'], categories=bin_order, ordered=True)

    celltypes_present = sorted(sub_obs[celltype_col].dropna().unique().tolist())
    n_ct = len(celltypes_present)
    ncols = 4
    nrows = (n_ct + ncols - 1) // ncols

    palette_bins = sns.color_palette('coolwarm', len(bin_order))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    for i, ct in enumerate(celltypes_present):
        ax = axes[i]
        ct_data = sub_obs[sub_obs[celltype_col] == ct]

        if len(ct_data) < 10:
            ax.set_title(f'{ct}\n(n<10, skip)', fontsize=8)
            ax.axis('off')
            continue

        # Spearman r
        r, p = spearmanr(ct_data['age_bin_numeric'].values, ct_data['_expr'].values)
        sig_str = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

        sns.boxplot(
            data=ct_data, x='age_bin', y='_expr',
            palette=palette_bins, order=bin_order,
            showfliers=False, ax=ax, linewidth=0.8
        )
        ax.set_title(f'{ct}\nr={r:.2f}, p={p:.3f} {sig_str}', fontsize=9, fontweight='bold')
        ax.set_xlabel('Age Bin', fontsize=8)
        ax.set_ylabel(f'{gene} Expression', fontsize=8)
        ax.tick_params(axis='x', rotation=45, labelsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    fig.suptitle(
        f'[{title_prefix}] {gene} — Age Bin Expression by Celltype',
        fontsize=14, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()


# Healthy / MASLD 각각 시각화
for group, adata_g in groups_corr.items():
    plot_gene_celltype_boxplot(adata_g, gene_of_interest, celltype_col, corr_bin_labels, group)

In [ ]:
# Gene-of-interest: celltype별 Spearman r 요약 테이블
def gene_celltype_corr_summary(gene, groups_corr, celltype_col, corr_bin_labels):
    rows = []
    for group, adata_g in groups_corr.items():
        if gene not in adata_g.var_names:
            continue
        idx = list(adata_g.var_names).index(gene)
        expr_vec = adata_g.X[:, idx]
        if sp.issparse(expr_vec):
            expr_vec = np.array(expr_vec.todense()).flatten()
        else:
            expr_vec = np.array(expr_vec).flatten()

        sub_obs = adata_g.obs.copy()
        sub_obs['_expr'] = expr_vec
        sub_obs = sub_obs[sub_obs['age_bin'].isin(corr_bin_labels)]

        for ct in sorted(sub_obs[celltype_col].dropna().unique()):
            ct_data = sub_obs[sub_obs[celltype_col] == ct]
            if len(ct_data) < 10:
                continue
            r, p = spearmanr(ct_data['age_bin_numeric'].values, ct_data['_expr'].values)
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            rows.append({'Disease_group': group, 'celltype': ct,
                         'spearman_r': round(r, 4), 'pval': round(p, 6),
                         'sig': sig, 'n_cells': len(ct_data)})
    df = pd.DataFrame(rows)
    return df

summary_df = gene_celltype_corr_summary(gene_of_interest, groups_corr, celltype_col, corr_bin_labels)
print(f"=== {gene_of_interest}: Celltype별 Spearman r Summary ===")
print(summary_df.to_string(index=False))
summary_df

## 8. Pearson Correlation Analysis

Section 6의 Spearman 분석과 동일한 구조로 **Pearson correlation** 을 수행합니다.  
`expr_dict`, `ages_dict`, `groups_corr`, `celltypes` 등 Section 6에서 생성된 변수를 재사용합니다.

- **8-1**: 전체 세포 — 실제 age 값 기반 Pearson r  
- **8-2**: age_bin numeric 기반 Pearson r  
- **8-3**: Celltype별 Pearson r  
- **8-4**: Pearson r Heatmap (celltype × gene)

### 8-1. 전체 세포 - 실제 age 값 기반 Pearson correlation

In [ ]:
# Correlation 분석용 age bin 필터: '-30' 과 '81-' 제외
exclude_bins = ['-30', '81-']
corr_bin_labels = [b for b in bin_labels if b not in exclude_bins]
print(f"Correlation 분석 age bin: {corr_bin_labels}")

# Disease_group별 필터된 서브셋 준비
groups_corr = {
    'Healthy': adata_healthy[~adata_healthy.obs['age_bin'].isin(exclude_bins)].copy(),
    'MASLD':   adata_masld[~adata_masld.obs['age_bin'].isin(exclude_bins)].copy(),
}

expr_dict = {}
ages_dict = {}

for group, adata_g in groups_corr.items():
    expr = adata_g.X
    if not isinstance(expr, np.ndarray):
        expr = expr.toarray()
    expr_dict[group] = expr
    ages_dict[group] = adata_g.obs['age_numeric'].values
    print(f"{group} — {adata_g.n_obs:,} 세포, age: {ages_dict[group].min():.0f}~{ages_dict[group].max():.0f}")

In [ ]:
# Healthy / MASLD 각각 age vs gene expression Pearson correlation
# (age bin: 31-40, 41-50, 51-60, 61-70, 71-80 만 포함)
age_pearson_results = {}

for group, adata_g in groups_corr.items():
    print(f"[{group}] Pearson correlation 계산 중...")
    expr = expr_dict[group]
    ages = ages_dict[group]
    var_names = adata_g.var_names

    corrs, pvals = [], []
    for i in range(expr.shape[1]):
        r, p = pearsonr(ages, expr[:, i])
        corrs.append(r)
        pvals.append(p)

    df_corr = pd.DataFrame({'gene': var_names, 'pearson_r': corrs, 'pval': pvals})
    age_pearson_results[group] = df_corr

    df_pos = df_corr[(df_corr['pearson_r'] > 0) & (df_corr['pval'] < 0.05)]
    df_neg = df_corr[(df_corr['pearson_r'] < 0) & (df_corr['pval'] < 0.05)]
    print(f"  significant: pos={len(df_pos)}, neg={len(df_neg)}")
    print(f"  Top 5 positive:\n{df_pos.nlargest(5, 'pearson_r')[['gene','pearson_r','pval']].to_string(index=False)}")
    print(f"  Top 5 negative:\n{df_neg.nsmallest(5, 'pearson_r')[['gene','pearson_r','pval']].to_string(index=False)}\n")

In [ ]:
output_dir = "/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/02_analysis/"

In [ ]:
# 전체 세포 Pearson correlation 결과 저장
for group, df_corr in age_pearson_results.items():
    path = output_dir + f"Nucseq_age_pearson_{group}.csv"
    df_corr.to_csv(path, index=False)
    print(f"저장 완료: {path}")

### 8-2. Age bin numeric 기반 Pearson correlation

In [ ]:
# ─────────────────────────────────────────────────────────────
# 8-2 독립 실행용 로드 셀 (8-1까지 완료 후 재시작 시 여기서부터)
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr

save_dir     = "/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/01_Data/"
output_dir   = "/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/02_analysis/"
celltype_col = 'celltype'

# corr_bin_labels 필터가 이미 적용된 h5ad 불러오기
adata_healthy_corr = sc.read_h5ad(save_dir + "Nucseq_Healthy_corr_bins.h5ad")
adata_masld_corr   = sc.read_h5ad(save_dir + "Nucseq_MASLD_corr_bins.h5ad")

groups_corr = {
    'Healthy': adata_healthy_corr,
    'MASLD':   adata_masld_corr,
}

expr_dict  = {}
ages_dict  = {}

for group, adata_g in groups_corr.items():
    expr = adata_g.X
    if not isinstance(expr, np.ndarray):
        expr = expr.toarray()
    expr_dict[group]  = expr
    ages_dict[group]  = adata_g.obs['age_numeric'].values
    print(f"{group} — {adata_g.n_obs:,} 세포, age: {ages_dict[group].min():.0f}~{ages_dict[group].max():.0f}")

print("\n로드 완료 — 8-2부터 실행 가능")

In [ ]:
# adata importing 
adata = sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad")
print(f"전체 세포 수: {adata.n_obs:,}, 유전자 수: {adata.n_vars:,}")
celltypes = sorted(adata.obs[celltype_col].dropna().unique().tolist())
print(f"분석할 celltype ({len(celltypes)}개): {celltypes}")

In [ ]:
# age_bin_numeric 기반 Pearson correlation — Healthy / MASLD
# (age bin: 31-40, 41-50, 51-60, 61-70, 71-80 만 포함)
bin_pearson_results = {}

for group, adata_g in groups_corr.items():
    print(f"[{group}] age_bin Pearson correlation 계산 중...")
    expr = expr_dict[group]
    age_bin_num = adata_g.obs['age_bin_numeric'].values
    var_names = adata_g.var_names

    corrs_bin, pvals_bin = [], []
    for i in range(expr.shape[1]):
        r, p = pearsonr(age_bin_num, expr[:, i])
        corrs_bin.append(r)
        pvals_bin.append(p)

    df_bin = pd.DataFrame({'gene': var_names, 'pearson_r': corrs_bin, 'pval': pvals_bin})
    bin_pearson_results[group] = df_bin

    df_pos = df_bin[(df_bin['pearson_r'] > 0) & (df_bin['pval'] < 0.05)]
    df_neg = df_bin[(df_bin['pearson_r'] < 0) & (df_bin['pval'] < 0.05)]
    print(f"  significant: pos={len(df_pos)}, neg={len(df_neg)}")
    print(f"  Top 5 positive:\n{df_pos.nlargest(5, 'pearson_r')[['gene','pearson_r','pval']].to_string(index=False)}")
    print(f"  Top 5 negative:\n{df_neg.nsmallest(5, 'pearson_r')[['gene','pearson_r','pval']].to_string(index=False)}\n")

In [ ]:
# age_bin_numeric 기반 Pearson correlation 결과 저장
for group, df_bin in bin_pearson_results.items():
    path = output_dir + f"Nucseq_age_bin_pearson_{group}.csv"
    df_bin.to_csv(path, index=False)
    print(f"저장 완료: {path}")

### 8-3. Celltype별 Age Pearson Correlation

In [ ]:
# celltype별 age Pearson correlation — Healthy / MASLD 각각
# (age bin: 31-40, 41-50, 51-60, 61-70, 71-80 만 포함)
celltype_pearson_results = {'Healthy': {}, 'MASLD': {}}

for group, adata_g in groups_corr.items():
    print(f"\n{'='*40}\n[{group}] celltype별 Pearson correlation 계산")
    for ct in celltypes:
        adata_ct = adata_g[adata_g.obs[celltype_col] == ct].copy()
        n_cells = adata_ct.n_obs
        if n_cells < 30:
            print(f"  [SKIP] {ct}: {n_cells}개 (< 30)")
            continue
        print(f"  [분석] {ct}: {n_cells:,}개")

        expr_ct = adata_ct.X
        if not isinstance(expr_ct, np.ndarray):
            expr_ct = expr_ct.toarray()
        ages_ct = adata_ct.obs['age_numeric'].values

        corrs_ct, pvals_ct = [], []
        for i in range(expr_ct.shape[1]):
            r, p = pearsonr(ages_ct, expr_ct[:, i])
            corrs_ct.append(r)
            pvals_ct.append(p)

        celltype_pearson_results[group][ct] = pd.DataFrame({
            'gene': adata_ct.var_names,
            'pearson_r': corrs_ct,
            'pval': pvals_ct
        })

print("\n분석 완료!")

In [ ]:
# Healthy / MASLD 각각 Top Pearson correlation gene 확인
for group in ['Healthy', 'MASLD']:
    print(f"\n{'='*50}\n[{group}]")
    for ct, df_ct in celltype_pearson_results[group].items():
        df_sig = df_ct[df_ct['pval'] < 0.05]
        n_pos = (df_sig['pearson_r'] > 0).sum()
        n_neg = (df_sig['pearson_r'] < 0).sum()
        print(f"\n  {ct}: sig={len(df_sig)} (pos:{n_pos}, neg:{n_neg})")
        print("  Top 5 pos:", df_sig[df_sig['pearson_r'] > 0].nlargest(5, 'pearson_r')['gene'].tolist())
        print("  Top 5 neg:", df_sig[df_sig['pearson_r'] < 0].nsmallest(5, 'pearson_r')['gene'].tolist())

In [ ]:
# celltype별 Pearson 결과 저장 — Healthy / MASLD 각각
for group in ['Healthy', 'MASLD']:
    for ct, df_ct in celltype_pearson_results[group].items():
        safe_ct = ct.replace('/', '_').replace(' ', '_')
        path = output_dir + f"Nucseq_age_pearson_{group}_{safe_ct}.csv"
        df_ct.to_csv(path, index=False)
        print(f"저장: {path}")

In [ ]:
# 8-3 통합 Excel 저장: Healthy / MASLD × 각 celltype Pearson 결과
import glob
import os
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter

RED_FILL  = PatternFill(start_color="FF9999", end_color="FF9999", fill_type="solid")
BLUE_FILL = PatternFill(start_color="99BBFF", end_color="99BBFF", fill_type="solid")

excel_path = output_dir + "Nucseq_age_pearson_celltype_integrated.xlsx"
wb = Workbook()
wb.remove(wb.active)

for group in ['Healthy', 'MASLD']:
    pattern = output_dir + f"Nucseq_age_pearson_{group}_*.csv"
    files = sorted(glob.glob(pattern))
    print(f"[{group}] {len(files)}개 파일 발견")

    all_dfs = []
    for f in files:
        basename = os.path.basename(f)
        ct = basename.replace(f"Nucseq_age_pearson_{group}_", "").replace(".csv", "")
        df = pd.read_csv(f)
        df.insert(0, 'celltype', ct)
        all_dfs.append(df)

    if not all_dfs:
        print(f"  [{group}] 파일 없음, 건너뜀")
        continue

    combined = pd.concat(all_dfs, ignore_index=True)
    ws = wb.create_sheet(title=group)

    headers = list(combined.columns)
    ws.append(headers)
    for cell in ws[1]:
        cell.font = Font(bold=True)

    for _, row in combined.iterrows():
        ws.append(list(row))
        current_row = ws.max_row
        pval  = row['pval']
        r_val = row['pearson_r']
        if pval < 0.05:
            fill = RED_FILL if r_val > 0 else (BLUE_FILL if r_val < 0 else None)
            if fill:
                for cell in ws[current_row]:
                    cell.fill = fill

    for col_idx, col_name in enumerate(headers, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = max(len(str(col_name)) + 4, 14)

    print(f"  [{group}] {len(combined)}행 작성 완료")

wb.save(excel_path)
print(f"\n저장 완료: {excel_path}")

### 8-4. Pearson r Heatmap: celltype x gene

In [ ]:
# Pearson r heatmap: celltype x gene — Healthy / MASLD 각각
for group in ['Healthy', 'MASLD']:
    ct_results = celltype_pearson_results[group]
    if len(ct_results) < 2:
        print(f"[{group}] celltype 수 부족, heatmap 생략")
        continue

    top_genes_all = set()
    for ct, df_ct in ct_results.items():
        top = df_ct[(df_ct['pearson_r'] > 0) & (df_ct['pval'] < 0.05)].nlargest(10, 'pearson_r')['gene'].tolist()
        top_genes_all.update(top)

    heatmap_data = {
        ct: df_ct.set_index('gene')['pearson_r'].reindex(list(top_genes_all))
        for ct, df_ct in ct_results.items()
    }
    heatmap_df = pd.DataFrame(heatmap_data).dropna(how='all')

    fig, ax = plt.subplots(figsize=(max(8, len(ct_results) * 1.5), max(6, len(heatmap_df) * 0.4)))
    sns.heatmap(
        heatmap_df, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
        ax=ax, linewidths=0.3, linecolor='lightgray',
        cbar_kws={'label': 'Pearson r'}
    )
    ax.set_title(f'[{group}] Age-Gene Pearson Correlation by Celltype', fontsize=13)
    ax.set_xlabel('Celltype', fontsize=11)
    ax.set_ylabel('Gene', fontsize=11)
    plt.tight_layout()
    plt.show()